# Ablation Studies and Negative Transfer

## Scientific objective
Compare representations, model families, task sharing, weighting, calibration, uncertainty filtering, split strategy, AD strata, and Tox21 endpoint subsets.

## Inputs
- Metrics from notebooks 10–20

## Expected outputs
- `results/ablations/ablation_matrix.csv`
- `results/ablations/negative_transfer.csv`

## Dependencies
pandas

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
An ablation is valid only when all non-ablated components and split assignments are held constant.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
Missing model artifacts are reported as not run, not imputed. Cross-notebook smoke results are pipeline checks rather than manuscript conclusions.

## Next notebook
[22_final_model_comparison.ipynb](./22_final_model_comparison.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [2]:
from toxicity_screening.utils import atomic_write_json
sources={
 "qsar":ROOT/"results/metrics/qsar_baselines.csv",
 "single_task_mlp":ROOT/"results/metrics/single_task_mlp.csv",
 "single_task_gnn":ROOT/"results/metrics/single_task_gnn.csv",
 "multitask_mlp":ROOT/"results/metrics/multitask_fingerprint.csv",
 "multitask_gnn":ROOT/"results/metrics/multitask_gnn.csv",
}
frames=[]
for family,path in sources.items():
    if path.exists():
        f=pd.read_csv(path); f["family"]=family
        if "partition" in f: f=f[f.partition=="test"]
        frames.append(f)
all_metrics=pd.concat(frames,ignore_index=True,sort=False)
all_metrics.to_csv(ROOT/"results/ablations/ablation_matrix.csv",index=False)
# Negative transfer is explicitly endpoint-specific.
transfer_path=ROOT/"results/ablations/multitask_transfer_fingerprint.csv"
if transfer_path.exists(): pd.read_csv(transfer_path).to_csv(ROOT/"results/ablations/negative_transfer.csv",index=False)
else: atomic_write_json({"status":"not_run"},ROOT/"results/ablations/negative_transfer_status.json")
display(all_metrics)

,endpoint,model,split_strategy,partition,n,positive_prevalence,threshold,roc_auc,pr_auc,mcc,...,nll,recall_at_precision_0.80,precision_at_recall_0.80,tn,fp,fn,tp,family,best_epoch,best_validation_loss
0,herg_blockade,dummy,global_scaffold,test,1883,0.507169,0.5,0.500000,0.507169,0.000000,...,0.693857,0.000000,0.507169,928,0,955,0,qsar,NaN,NaN
1,herg_blockade,logistic_regression,global_scaffold,test,1883,0.507169,0.5,0.740934,0.746468,0.358241,...,1.469405,0.294241,0.629325,619,309,295,660,qsar,NaN,NaN
2,herg_blockade,random_forest,global_scaffold,test,1883,0.507169,0.5,0.827999,0.843768,0.505120,...,0.510772,0.680628,0.723324,749,179,291,664,qsar,NaN,NaN
3,herg_blockade,svm,global_scaffold,test,1883,0.507169,0.5,0.828030,0.841841,0.479565,...,0.511140,0.628272,0.711770,709,219,272,683,qsar,NaN,NaN
4,herg_blockade,gradient_boosting,global_scaffold,test,1883,0.507169,0.5,0.812583,0.831857,0.450427,...,0.528877,0.595812,0.697080,703,225,294,661,qsar,NaN,NaN
5,herg_blockade,xgboost,global_scaffold,test,1883,0.507169,0.5,0.813543,0.833711,0.452102,...,0.526184,0.616754,0.694545,710,218,300,655,qsar,NaN,NaN
6,ames_mutagenicity,dummy,global_scaffold,test,1120,0.547321,0.5,0.500000,0.547321,0.000000,...,0.689210,0.000000,0.547321,0,507,0,613,qsar,NaN,NaN
7,ames_mutagenicity,logistic_regression,global_scaffold,test,1120,0.547321,0.5,0.722788,0.732103,0.338604,...,2.679486,0.314845,0.672603,330,177,191,422,qsar,NaN,NaN
8,ames_mutagenicity,random_forest,global_scaffold,test,1120,0.547321,0.5,0.865463,0.890525,0.563414,...,0.463950,0.802610,0.809211,382,125,117,496,qsar,NaN,NaN
9,ames_mutagenicity,svm,global_scaffold,test,1120,0.547321,0.5,0.865263,0.892138,0.571802,...,0.455360,0.812398,0.804918,392,115,123,490,qsar,NaN,NaN


### Completion gate
Confirm that the declared artifacts exist before continuing to `22_final_model_comparison.ipynb`.